### Hybrid labeling

From Pre Labeling we got 803 reviews to manualy review and check the label.It's Not easy , so we will reduce that.

We will now reduce the manual review workload by combining:
- High-confidence zero-shot labels
- Keyword rules for obvious reviews
- A generic-review filter
- A smaller manual review list



Load libraries and data

In [1]:
import pandas as pd
import re
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

df_neg = pd.read_csv(PROCESSED_DIR / "negative_pre_labeled.csv")
df_pos = pd.read_csv(PROCESSED_DIR / "positive_pre_labeled.csv")

print(f"Negative reviews loaded: {len(df_neg)}")
print(f"Positive reviews loaded: {len(df_pos)}")

Negative reviews loaded: 745
Positive reviews loaded: 349


Define keyword rules

In [2]:
NEGATIVE_KEYWORDS = {
    "App Bugs": [
        "app crash", "crash", "crashed", "crashing",
        "bug", "bugs", "glitch", "error",
        "freeze", "frozen", "stuck",
        "update", "login", "log in", "sign in",
        "otp", "map", "gps", "location",
        "payment fail", "payment failed", "card",
        "server", "technical",
        "slow app", "app slow", "app is slow",
        "not working", "does not work", "app does not work"
    ],

    "Pricing & Bidding": [
        "price", "prices", "pricing",
        "fare", "fares",
        "rate", "rates",
        "charge", "charged", "charging",
        "expensive", "cost", "costly",
        "surge", "bid", "bidding",
        "offer", "offers",
        "discount", "discounts",
        "promo", "coupon",
        "high fare", "high price", "too high",
        "toll"
    ],

    "Delivery Issues": [
        "food", "restaurant", "meal", "meals",
        "order", "orders",
        "delivery", "deliver", "delivered",
        "late", "delayed",
        "missing item", "missing items",
        "wrong item", "wrong order",
        "cold food",
        "item missing", "items missing"
    ],

    "Driver Behavior & Safety": [
        "driver", "drivers",
        "rude", "unsafe", "dangerous", "reckless",
        "accident", "harass", "harassed",
        "behaviour", "behavior",
        "driving",
        "no show", "no-show",
        "did not come", "didn't come",
        "driver cancel", "driver cancelled"
    ],

    "Support & Refunds": [
        "support", "customer service", "customer care",
        "service center",
        "refund", "refunds",
        "ticket", "tickets",
        "hotline", "call center",
        "response", "respond", "responded",
        "ignore", "ignored",
        "complaint", "complaints",
        "no response", "not responding"
    ],

    "Privacy & Security": [
        "privacy",
        "personal data", "personal information",
        "data leak",
        "hacked", "hack",
        "security",
        "unauthorized", "unauthorised",
        "fraud", "scam",
        "account hacked", "account security"
    ]
}

POSITIVE_KEYWORDS = {
    "App Experience": [
        "app", "application",
        "interface", "ui", "ux",
        "user friendly", "user-friendly",
        "easy to use", "easy app",
        "smooth",
        "fast app", "app is fast",
        "good app", "nice app",
        "love app", "great app",
        "excellent app", "amazing app",
        "helpful app", "help full",
        "very helpful", "convenient"
    ],

    "Fair Pricing": [
        "price", "prices", "pricing",
        "fare", "fares",
        "rate", "rates",
        "cheap", "affordable", "reasonable",
        "value",
        "discount", "discounts",
        "promo", "coupon",
        "low price", "fair price",
        "transparent"
    ],

    "Fast Delivery": [
        "delivery", "deliver", "delivered",
        "food", "restaurant", "order", "meal",
        "on time", "ontime", "timely",
        "fast delivery", "quick delivery",
        "hot food", "fresh"
    ],

    "Driver Service": [
        "driver", "drivers", "rider",
        "polite", "friendly", "courteous",
        "professional",
        "safe", "safety",
        "careful",
        "good driver", "nice driver",
        "excellent driver",
        "kind"
    ],

    "Good Support": [
        "support", "customer service", "customer care",
        "helpful support",
        "responsive", "quick response",
        "resolved",
        "refund",
        "service"
    ],

    "Trust & Security": [
        "privacy",
        "secure", "security",
        "trust",
        "safe payment", "safe payments",
        "account safety"
    ]
}

Keyword matching function

This function checks how many keywords match each category.If one category has the highest keyword score, it assigns that category.If multiple categories tie, it marks the review as ambiguous.

In [4]:
def assign_keyword_label(text, keyword_dict):

    text = str(text).lower()

    # Initialize a dictionary to store the scores for each category
    scores = {}

    # Iterate over each category and its associated keywords
    for category, keywords in keyword_dict.items():
        # Initialize the score for the category
        score = 0

        # Iterate over each keyword in the category
        for keyword in keywords:
            # Create a regular expression pattern for the keyword
            pattern = r"\b" + re.escape(keyword.lower()) + r"\b"

            # If the pattern is found in the text, increment the score
            if re.search(pattern, text):
                score += 1

        # Store the score for the category
        scores[category] = score

    # Find the maximum score
    max_score = max(scores.values())

    # If no keyword is found, return None and the scores
    if max_score == 0:
        return None, scores

    # Find the top categories with the maximum score
    top_categories = [
        category for category, score in scores.items()
        if score == max_score
    ]

    # If there is a unique match, return the top category and the scores
    if len(top_categories) == 1:
        return top_categories[0], scores

    # If there is a tie, return 'AMBIGUOUS' and the scores
    return "AMBIGUOUS", scores

Apply hybrid labeling

In [5]:
AUTO_THRESHOLD = 0.65
GENERIC_LENGTH_THRESHOLD = 25

def refine_dataframe(df, keyword_dict):

    # Make a copy of the DataFrame
    df = df.copy()

    # Add a new column to store the length of the review text
    df["review_length"] = df["review_text"].str.len()

    # Add new columns to store the final category and label method
    df["final_category"] = None
    df["label_method"] = None

    # 1. Keep high-confidence zero-shot labels
    high_confidence_mask = df["category_confidence"] >= AUTO_THRESHOLD

    # Assign high-confidence labels to the corresponding rows
    df.loc[high_confidence_mask, "final_category"] = df.loc[high_confidence_mask, "category"]
    df.loc[high_confidence_mask, "label_method"] = "auto_high_confidence"

    # 2. Process low and medium confidence reviews
    low_medium_df = df[~high_confidence_mask]

    # Iterate over each row in the low and medium confidence DataFrame
    for idx, row in low_medium_df.iterrows():
        # Assign a label based on keyword rules
        keyword_label, keyword_scores = assign_keyword_label(
            row["review_text"],
            keyword_dict
        )

        if keyword_label not in [None, "AMBIGUOUS"]:
            df.at[idx, "final_category"] = keyword_label
            df.at[idx, "label_method"] = "keyword_rule"

        # Filter out reviews that are too generic
        elif row["review_length"] < GENERIC_LENGTH_THRESHOLD:
            df.at[idx, "final_category"] = "Exclude / Too Generic"
            df.at[idx, "label_method"] = "generic_filter"

        # Mark reviews for manual review if they are not covered by keyword rules or are too generic
        else:
            df.at[idx, "final_category"] = "Manual Review Needed"
            df.at[idx, "label_method"] = "manual_review_needed"

    return df

# Refine the negative DataFrame using the NEGATIVE_KEYWORDS dictionary
df_neg_refined = refine_dataframe(df_neg, NEGATIVE_KEYWORDS)

# Refine the positive DataFrame using the POSITIVE_KEYWORDS dictionary
df_pos_refined = refine_dataframe(df_pos, POSITIVE_KEYWORDS)

In [6]:
def summarize_refined(df, name):
    print(f"\n--- {name} ---")

    print("\nLabel method counts:")
    print(df["label_method"].value_counts())

    print("\nFinal category counts:")
    print(df["final_category"].value_counts())

    manual_count = (df["label_method"] == "manual_review_needed").sum()
    print(f"\nManual review needed: {manual_count}")

summarize_refined(df_neg_refined, "Negative Reviews")
summarize_refined(df_pos_refined, "Positive Reviews")


--- Negative Reviews ---

Label method counts:
label_method
keyword_rule            259
manual_review_needed    236
auto_high_confidence    197
generic_filter           53
Name: count, dtype: int64

Final category counts:
final_category
Manual Review Needed        236
Driver Behavior & Safety    166
App Bugs                     80
Delivery Issues              71
Pricing & Bidding            62
Support & Refunds            61
Exclude / Too Generic        53
Privacy & Security           16
Name: count, dtype: int64

Manual review needed: 236

--- Positive Reviews ---

Label method counts:
label_method
keyword_rule            127
auto_high_confidence     94
manual_review_needed     66
generic_filter           62
Name: count, dtype: int64

Final category counts:
final_category
App Experience           101
Manual Review Needed      66
Exclude / Too Generic     62
Driver Service            61
Good Support              35
Fair Pricing              15
Fast Delivery              9
Name: count,

Inspect keyword-rule examples

In [7]:
def show_keyword_samples(df, name, samples_per_category=2):

    # Print the name and category samples header
    print(f"\n--- {name}: keyword_rule samples ---")

    # Filter the DataFrame to only include rows where the label method is "keyword_rule"
    keyword_df = df[df["label_method"] == "keyword_rule"]

    # Iterate over each unique category in the filtered DataFrame
    for category in sorted(keyword_df["final_category"].unique()):
        # Filter the DataFrame to only include rows for the current category
        category_df = keyword_df[keyword_df["final_category"] == category]

        # Determine the number of samples to print for the current category
        sample_size = min(samples_per_category, len(category_df))

        # Randomly sample the specified number of rows from the current category DataFrame
        sample_df = category_df.sample(sample_size, random_state=42)

        # Print the category name and total number of labeled reviews for the category
        print(f"\n{category} | total keyword labels: {len(category_df)}")

        # Print a sample of the labeled reviews for the category
        for _, row in sample_df.iterrows():
            # Replace newlines with spaces in the review text
            text = str(row["review_text"]).replace("\n", " ")

            # Print the review text, truncated to 180 characters
            print(f"- {text[:180]}")

# Print keyword-rule labeled samples from the negative DataFrame
show_keyword_samples(df_neg_refined, "Negative Reviews")

# Print keyword-rule labeled samples from the positive DataFrame
show_keyword_samples(df_pos_refined, "Positive Reviews")


--- Negative Reviews: keyword_rule samples ---

App Bugs | total keyword labels: 68
- While the travel they blocked the app now. I need an update. but I don't have enough space. I faced severe problem
- good but need a system update.

Delivery Issues | total keyword labels: 55
- too slow, to have food arrive, we should at least book it 5 hrs before.
- Worst experience with the PickMe app. I placed an order at 6:00 PM, expecting a timely delivery, but it wasn't delivered until 10:30 PM. That's a delay of more than 4 hours, which 

Driver Behavior & Safety | total keyword labels: 59
- Worst app of all time When I wanted my drivers details they cause I need to know if my child arrived at the class. They said wait 24 hours to get the details , so that mean I need 
- Interest in using this app is decreasing. It would be much better to conduct a face-to-face interview before selecting drivers, or at least allow only people within a suitable age 

Pricing & Bidding | total keyword labels: 43

After sample chack I noticed some negative complaints misclassified in project 01 as positive. Before export manual review file, should clean this errors.

- Filter out reviews with low sentiment confidence.
- Use keyword rules to detect likely sentiment errors.
- Remove those errors from the dataset.

In [8]:
# Define sentiment confidence threshold
SENTIMENT_CONF_THRESHOLD = 0.7

# Filter out low confidence sentiment reviews
df_neg_refined = df_neg_refined[df_neg_refined["confidence"] >= SENTIMENT_CONF_THRESHOLD].copy()
df_pos_refined = df_pos_refined[df_pos_refined["confidence"] >= SENTIMENT_CONF_THRESHOLD].copy()

print(f"Negative reviews after sentiment confidence filter: {len(df_neg_refined)}")
print(f"Positive reviews after sentiment confidence filter: {len(df_pos_refined)}")

# Define sentiment keywords for sanity check
NEGATIVE_SENTIMENT_KEYWORDS = [
    "bug", "crash", "late", "rude", "expensive", "scam", "worst", "bad", "poor", "slow",
    "delay", "cancel", "refund", "complaint", "problem", "issue", "error", "fail",
    "disappointed", "frustrating", "useless", "terrible", "horrible", "awful",
    "takes ages", "never again", "not working", "waste of time", "👎"
]

POSITIVE_SENTIMENT_KEYWORDS = [
    "good", "great", "excellent", "love", "amazing", "friendly", "polite", "fast",
    "easy", "helpful", "best", "perfect", "awesome", "wonderful", "fantastic",
    "recommend", "happy", "satisfied", "thank you", "thanks", "👍", "❤️"
]

def check_sentiment_error(text, expected_sentiment):
    text = str(text).lower()
    
    neg_count = sum(1 for kw in NEGATIVE_SENTIMENT_KEYWORDS if kw in text)
    pos_count = sum(1 for kw in POSITIVE_SENTIMENT_KEYWORDS if kw in text)
    
    if expected_sentiment == "Positive":
        return neg_count > pos_count
    elif expected_sentiment == "Negative":
        return pos_count > neg_count
    return False

# Apply sentiment sanity check
df_pos_refined["sentiment_error"] = df_pos_refined["review_text"].apply(lambda x: check_sentiment_error(x, "Positive"))
df_neg_refined["sentiment_error"] = df_neg_refined["review_text"].apply(lambda x: check_sentiment_error(x, "Negative"))

print(f"\nPositive sentiment errors found: {df_pos_refined['sentiment_error'].sum()}")
print(f"Negative sentiment errors found: {df_neg_refined['sentiment_error'].sum()}")

# Filter out sentiment errors
df_pos_refined = df_pos_refined[~df_pos_refined["sentiment_error"]].copy()
df_neg_refined = df_neg_refined[~df_neg_refined["sentiment_error"]].copy()

print(f"\nNegative reviews after sentiment sanity check: {len(df_neg_refined)}")
print(f"Positive reviews after sentiment sanity check: {len(df_pos_refined)}")

# Drop the sentiment_error column
df_pos_refined = df_pos_refined.drop(columns=["sentiment_error"])
df_neg_refined = df_neg_refined.drop(columns=["sentiment_error"])

Negative reviews after sentiment confidence filter: 736
Positive reviews after sentiment confidence filter: 343

Positive sentiment errors found: 8
Negative sentiment errors found: 52

Negative reviews after sentiment sanity check: 684
Positive reviews after sentiment sanity check: 335


In [9]:
def export_refined_and_manual_files(df, prefix):
    df = df.copy()

    # 1. Save full refined dataset
    refined_path = PROCESSED_DIR / f"{prefix}_refined.csv"
    df.to_csv(refined_path, index=False)

    # 2. Save accepted labels
    accepted = df[df["label_method"].isin(["auto_high_confidence", "keyword_rule"])].copy()
    accepted = accepted.rename(columns={"final_category": "label"})

    accepted_path = PROCESSED_DIR / f"{prefix}_accepted_labels.csv"
    accepted.to_csv(accepted_path, index=False)

    # 3. Save manual review file
    manual = df[df["label_method"] == "manual_review_needed"].copy()

    manual_columns = [
        "review_text",
        "rating",
        "date",
        "sentiment",
        "category",
        "category_confidence",
        "confidence_band",
        "review_length"
    ]

    manual_columns = [col for col in manual_columns if col in manual.columns]

    manual = manual[manual_columns].copy()
    manual = manual.rename(columns={"category": "suggested_category"})
    manual = manual.sort_values("category_confidence")
    manual["corrected_category"] = ""

    manual_path = PROCESSED_DIR / f"{prefix}_manual_review.csv"
    manual.to_csv(manual_path, index=False)

    # 4. Save excluded generic reviews
    excluded = df[df["label_method"] == "generic_filter"].copy()

    excluded_path = PROCESSED_DIR / f"{prefix}_excluded_generic.csv"
    excluded.to_csv(excluded_path, index=False)

    print(f"\n--- {prefix.upper()} ---")
    print(f"Accepted labels: {len(accepted)}")
    print(f"Manual review: {len(manual)}")
    print(f"Excluded generic: {len(excluded)}")
    print(f"Saved: {refined_path}")
    print(f"Saved: {accepted_path}")
    print(f"Saved: {manual_path}")
    print(f"Saved: {excluded_path}")

export_refined_and_manual_files(df_neg_refined, "negative")
export_refined_and_manual_files(df_pos_refined, "positive")


--- NEGATIVE ---
Accepted labels: 429
Manual review: 212
Excluded generic: 43
Saved: ..\data\processed\negative_refined.csv
Saved: ..\data\processed\negative_accepted_labels.csv
Saved: ..\data\processed\negative_manual_review.csv
Saved: ..\data\processed\negative_excluded_generic.csv

--- POSITIVE ---
Accepted labels: 211
Manual review: 63
Excluded generic: 61
Saved: ..\data\processed\positive_refined.csv
Saved: ..\data\processed\positive_accepted_labels.csv
Saved: ..\data\processed\positive_manual_review.csv
Saved: ..\data\processed\positive_excluded_generic.csv
